# How to Build Multiple Matrices Without Repeating Work
> **Set up**
>
> To run this notebook, first install the Julia kernel for Jupyter Notebooks using [IJulia](https://julialang.github.io/IJulia.jl/stable/manual/installation/), then [create an environment](https://pkgdocs.julialang.org/v1/environments/) for this tutorial with the packages listed with `using <PackageName>` further down.
>
> This tutorial has demonstrated compatibility with these package versions. If you run into any errors, first check your package versions for consistency using `Pkg.status()`.
>
 > ```
 > Status `~/work/PowerNetworkMatrices.jl/PowerNetworkMatrices.jl/docs/Project.toml`
 >   [a93c6f00] DataFrames v1.8.2
 >   [864edb3b] DataStructures v0.19.6
 >   [e30172f5] Documenter v1.17.0
 >   [d12716ef] DocumenterInterLinks v1.1.0
 >   [98b081ad] Literate v2.21.0
 >   [bed98974] PowerNetworkMatrices v0.24.3 `~/work/PowerNetworkMatrices.jl/PowerNetworkMatrices.jl`
 >   [f00506e0] PowerSystemCaseBuilder v2.5.0
 >   [bcd98974] PowerSystems v5.12.0
 >   [08abe8d2] PrettyTables v3.4.3
 > 
 > ```



Every matrix constructor that takes a `System`
rebuilds the same intermediates from scratch — the `Ybus`, the incidence
matrix `A` (`IncidenceMatrix`), and the susceptance-weighted `BA`
(`BA_Matrix`). This guide shows how to compute the shared pieces once and
feed them to the constructors that accept pre-built matrices.

In [ ]:
using PowerNetworkMatrices
import PowerSystemCaseBuilder as PSB

sys = PSB.build_system(PSB.PSITestSystems, "c_sys5");

## Build the shared intermediates once

The construction dependency chain is

> `Ybus` → `IncidenceMatrix`, `BA_Matrix` →
> `ABA_Matrix` / `PTDF`, and `PTDF` → `LODF`.

The `Ybus` is the expensive shared root. Build it — and the incidence and
BA matrices derived from it — exactly once:

In [ ]:
ybus = Ybus(sys)
A = IncidenceMatrix(ybus)
BA = BA_Matrix(ybus)

## Reuse them across constructors

`PTDF` accepts the incidence and BA matrices directly, skipping its own
`Ybus` build:

In [ ]:
ptdf = PTDF(A, BA)

`LODF` can be built straight from a `PTDF` you already have,
reusing that work too — no second factorization of the network:

In [ ]:
lodf = LODF(A, ptdf)

Alternatively, the factorized `ABA_Matrix` route builds `LODF`
from the same `A` and `BA`. All three inputs must share the same network
reduction — which they do here, because they all descend from one `ybus`:

In [ ]:
aba = ABA_Matrix(ybus; factorize = true)
lodf_via_aba = LODF(A, aba, BA)

Virtual matrices likewise accept a pre-built `Ybus`, so the lazy forms
reuse the same root:

In [ ]:
vptdf = VirtualPTDF(ybus)

> *Keep reductions consistent*
>
>
> Constructors that combine pre-built matrices (e.g. `LODF(A, ABA, BA)`) require
> every input to have been built with the **same** `network_reductions`. Because
> they all derive from a single `Ybus` here, they are automatically
> consistent. Pass `network_reductions` once, to the `Ybus` call, and
> everything downstream inherits it. See the `NetworkReduction` docstring
> for the keyword and its rules.

## See also

  - Matrix overview & indexing — every matrix type, its axes, and how the
    shared intermediates fit together.
  - How to Choose a Linear Solver — the factorization cost that reuse
    avoids repeating.
  - `NetworkReduction` — supplying reductions via `network_reductions`
    to the shared `Ybus`.